In [ ]:
import pandas as pd
import numpy as np
import os
from model_tuner import loadObjects

from core.constants import drop_vars

## Paths

In [ ]:
data_path = "../data/processed"

## Load the best models from the dramatic_model experiment.

In [ ]:
# Best overall -- CB Feats+Text, AUC 0.815 (dramatic_text_model experiment)
model_cat_feats_and_text = loadObjects(
    "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/243220405481969524/a238e602049a4997971e7dd1ba803949/artifacts/cat_feats_and_text_dramatic/model.pkl"
)

# Best text only -- CB Text Only, AUC 0.709 (dramatic_text_model experiment)
model_cat_text_only = loadObjects(
    "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/243220405481969524/449855b0e7e64982b4154669a3da6a57/artifacts/cat_text_only_dramatic/model.pkl"
)

# Best tabular CatBoost -- cat smote, AUC 0.806 (dramatic_model experiment)
model_cat = loadObjects(
    "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/431105840581859661/a2fdc9def5df44b7bb47bd37e078b406/artifacts/cat_dramatic/model.pkl"
)

# Best LR -- lr orig, AUC 0.780 (dramatic_model experiment)
model_lr = loadObjects(
    "/home/lshpaner/Python_Projects/nuforc_media/mlruns/models/431105840581859661/1b5777f73afd46928d6f8c1327e66ff2/artifacts/lr_dramatic/model.pkl"
)

## Load test data for final evaluation of best models

In [ ]:
X = pd.read_parquet(os.path.join(data_path, "X.parquet"))
y = pd.read_parquet(os.path.join(data_path, "y_dramatic.parquet"))

In [ ]:
X_full = pd.read_parquet("../data/processed/X.parquet")
y = pd.read_parquet("../data/processed/y_dramatic.parquet").squeeze()

# cat_feats_and_text
X_feats_and_text = X_full.drop(columns=[c for c in drop_vars if c in X_full.columns], errors="ignore")
X_feats_and_text["summary_clean"] = X_feats_and_text["summary_clean"].fillna("").astype(str)

# cat_text_only
X_text_only = X_full[["summary_clean"]].copy()
X_text_only["summary_clean"] = X_text_only["summary_clean"].fillna("").astype(str)

# tabular (cat, lr)
X_tabular = X_full.drop(columns=["summary_clean"], errors="ignore")

# Now pull test splits from each model
X_test_feats_and_text, y_test = model_cat_feats_and_text.get_test_data(X_feats_and_text, y)
X_test_text_only, _           = model_cat_text_only.get_test_data(X_text_only, y)
X_test_cat, _                 = model_cat.get_test_data(X_tabular, y)
X_test_lr, _                  = model_lr.get_test_data(X_tabular, y)

y_prob_cat_feats_and_text = model_cat_feats_and_text.predict_proba(X_test_feats_and_text)[:, 1]
y_prob_cat_text_only      = model_cat_text_only.predict_proba(X_test_text_only)[:, 1]
y_prob_cat                = model_cat.predict_proba(X_test_cat)[:, 1]
y_prob_lr                 = model_lr.predict_proba(X_test_lr)[:, 1]

In [ ]:
# X_test, y_test = model_lr.get_test_data(X,y)

In [ ]:
pd.DataFrame(model_cat_feats_and_text.get_feature_names(), columns=["Features"])

In [ ]:
len(model_cat_feats_and_text.get_feature_names())

## ROC AUC and Precision-Recall AUC for Regular Features Plus Combined Text

In [ ]:
from model_metrics import combine_plots
combine_plots(
    plot_calls=[
        (
            show_roc_curve,
            {
                "y_prob": [
                    y_prob_cat_feats_and_text,
                    y_prob_cat_text_only,
                    y_prob_cat,
                    y_prob_lr,
                ],
                "y": y_test,
                "model_title": [
                    "CatBoost Feats + Text",
                    "CatBoost Text Only",
                    "CatBoost Tabular (SMOTE)",
                    "Logistic Regression",
                ],
                "overlay": True,
                "legend_loc": "bottom",
                "decimal_places": 2,
            },
        ),
        (
            show_pr_curve,
            {
                "y_prob": [
                    y_prob_cat_feats_and_text,
                    y_prob_cat_text_only,
                    y_prob_cat,
                    y_prob_lr,
                ],
                "y": y_test,
                "model_title": [
                    "CatBoost Feats + Text",
                    "CatBoost Text Only",
                    "CatBoost Tabular (SMOTE)",
                    "Logistic Regression",
                ],
                "overlay": True,
                "legend_loc": "bottom",
                "decimal_places": 2,
            },
        ),
    ],
    n_cols=2,
    suptitle="All Models: ROC & PR Curves (Dramatic Outcome)",
    suptitle_y=1.06,
    hspace=0.05,
)

In [ ]:
X_test.columns.to_list()

In [ ]:
from eda_toolkit import flex_corr_matrix

flex_corr_matrix(
    df=X_test,
    cols=X_test.columns.to_list(),
    annot=True,
    cmap="coolwarm",
    figsize=(10, 8),
    title="US Census Correlation Matrix",
    xlabel_alignment="right",
    label_fontsize=8,
    tick_fontsize=8,
    xlabel_rot=45,
    ylabel_rot=0,
    text_wrap=50,
    vmin=-1,
    vmax=1,
    cbar_label="Correlation Index",
    triangular=True,
)

In [ ]:
X_test["cluster_id"].value_counts()

In [ ]:
[col for col in X_test.columns if "lat" in col]